<a href="https://colab.research.google.com/github/elisabethcoller/CDD-Noise-Filtering-Algorithm/blob/main/microcontroller_algorithm/my_algorithm_10_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time #pause execution of loop
import os #check csv
import pandas as pd #process tabular data
import numpy as np #mathematical functions

# =====================================================================
# CONFIGURATION FOR MICRO-BUFFERED STREAMER WITH ACTUAL DATA
# =====================================================================
BUFFER_CONFIG = {
    "file_path": "s3.csv",
    "sampling_rate_hz": 256,
    "speed_factor": 1.0,        # Increase to run simulation faster than real-time
    "buffer_size_samples": 20, # Set buffer size to exactly 20 samples as requested
    "limit_seconds": 2.0,       # Run for a short duration to demonstrate the output

    # Column mapping matching your s3.csv dataset
    "column_mapping": {
        "time": "sample_times_for_all_signals_apart_from_ecg",
        "ppg": "wrist_ppg",
        "gyro_x": "wrist_gyro_x",
        "gyro_y": "wrist_gyro_y",
        "gyro_z": "wrist_gyro_z",
        "accel_x": "wrist_low_noise_accelerometer_x",
        "accel_y": "wrist_low_noise_accelerometer_y",
        "accel_z": "wrist_low_noise_accelerometer_z"
    }
}

def stream_with_micro_buffering(config):
    path = config["file_path"]
    rate = config["sampling_rate_hz"]
    buffer_size = config["buffer_size_samples"]
    speed = config["speed_factor"]
    limit_seconds = config["limit_seconds"]
    cols = config["column_mapping"]

    # Load actual data
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"[✓] Loaded data from {path} ({len(df)} rows)")
    else:
        print(f"[!] File {path} not found. Generating mock data...")
        total_samples = int(limit_seconds * rate)
        times = np.linspace(0, limit_seconds, total_samples)
        df = pd.DataFrame({
            cols["time"]: times,
            cols["ppg"]: np.sin(2 * np.pi * 1.2 * times),
            cols["gyro_x"]: np.zeros(total_samples),
            cols["gyro_y"]: np.zeros(total_samples),
            cols["gyro_z"]: np.zeros(total_samples),
            cols["accel_x"]: np.zeros(total_samples),
            cols["accel_y"]: np.zeros(total_samples),
            cols["accel_z"]: np.zeros(total_samples)
        })

    # Calculate simulated physical time duration that one buffer represents
    # For 20 samples at 256Hz, this is 20 / 256 = ~0.078 seconds of real-world data
    buffer_duration_seconds = buffer_size / rate
    sleep_delay = buffer_duration_seconds / speed

    total_samples_to_run = int(limit_seconds * rate)

    print(f"\n[Micro-Buffering Simulation]")
    print(f"- Each buffer holds: {buffer_size} samples")
    print(f"- Buffer duration in real-world time: {buffer_duration_seconds:.4f} seconds")
    print(f"- Simulator sleeps for {sleep_delay:.4f}s between buffers (Speed: {speed}x)\n")

    try:#run the code in this section until an error is found
        buffer_count = 0
        for start_idx in range(0, min(len(df), total_samples_to_run), buffer_size): #range(start,stop,step)
            end_idx = start_idx + buffer_size
            buffer_df = df.iloc[start_idx:end_idx] #selects rows or columns by integer position

            buffer_count += 1
            print(f"==================== [⚡ Buffer #{buffer_count} Received] ====================")

            # Handle and print every individual value in this batch sequentially
            for idx, row in buffer_df.iterrows():
                t = row[cols["time"]]
                ppg = row[cols["ppg"]]
                gx, gy, gz = row[cols["gyro_x"]], row[cols["gyro_y"]], row[cols["gyro_z"]]
                ax, ay, az = row[cols["accel_x"]], row[cols["accel_y"]], row[cols["accel_z"]]

                print(f"  [Sample Index: {idx:04d} | Time: {t:.4f}s] ")
                print(f"     PPG  : {ppg:.2f}")
                print(f"     Gyro : X={gx:.2f}, Y={gy:.2f}, Z={gz:.2f}")
                print(f"     Accel: X={ax:.2f}, Y={ay:.2f}, Z={az:.2f}")
                print(f"     -" * 20)

            time.sleep(sleep_delay)

    except KeyboardInterrupt:
        print("\nBuffer stream stopped.")

# Run the updated micro-buffered simulation
stream_with_micro_buffering(BUFFER_CONFIG)

[✓] Loaded data from s3.csv (78593 rows)

[Micro-Buffering Simulation]
- Each buffer holds: 20 samples
- Buffer duration in real-world time: 0.0781 seconds
- Simulator sleeps for 0.0781s between buffers (Speed: 1.0x)

==================== [⚡ Buffer #1 Received] ====================
  [Sample Index: 0000 | Time: 24.9957s] 
     PPG  : 1458.60
     Gyro : X=8.88, Y=-78.61, Z=-5.64
     Accel: X=1.51, Y=-7.79, Z=1.09
     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -
  [Sample Index: 0001 | Time: 25.0003s] 
     PPG  : 1443.98
     Gyro : X=8.19, Y=-79.26, Z=-5.40
     Accel: X=1.60, Y=-7.84, Z=1.10
     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -     -
  [Sample Index: 0002 | Time: 25.0040s] 
     PPG  : 1431.49
     Gyro : X=7.18, Y=-78.97, Z=-5.07
     Accel: X=1.69, Y=-7.91, Z=1.09
     -     -     -     -     -     -     -     -     -     -     -    